In [ ]:
from kafka import KafkaConsumer
from collections import defaultdict, deque
from datetime import datetime, timedelta
import json
import time
import clickhouse_connect

CLICKHOUSE_HOST = 'clickhouse'

try:
    ch_client = clickhouse_connect.get_client(
        host=CLICKHOUSE_HOST,
        port=8123,
        username='admin',
        password='67'
    )
    print("Pomyślnie połączono z ClickHouse.")
except Exception as e:
    print(f"\n[BŁĄD FATALNY] Brak połączenia z ClickHouse na hoście '{CLICKHOUSE_HOST}':\n{e}")
    raise

# Konfiguracja konsumenta Kafki
konsument = KafkaConsumer(
    'bank-transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')),
    group_id='risk-analysis-consumer'
)

transactions = []
user_activity = defaultdict(deque)

polskie_miasta = [
    'Warszawa', 'Kraków', 'Łódź', 'Wrocław', 'Poznań',
    'Gdańsk', 'Szczecin', 'Bydgoszcz', 'Lublin', 'Białystok',
    'Katowice', 'Gdynia', 'Częstochowa', 'Radom', 'Sosnowiec',
    'Toruń', 'Kielce', 'Rzeszów', 'Gliwice', 'Zabrze',
    'Olsztyn', 'Bielsko-Biała', 'Bytom', 'Zielona Góra', 'Rybnik',
    'Ruda Śląska', 'Tychy', 'Gorzów Wielkopolski', 'Dąbrowa Górnicza', 'Elbląg',
    'Płock', 'Opole', 'Wałbrzych', 'Włocławek', 'Tarnów',
    'Chorzów', 'Koszalin', 'Kalisz', 'Legnica', 'Grudziądz',
    'Jaworzno', 'Słupsk', 'Jastrzębie-Zdrój', 'Nowy Sącz', 'Jelenia Góra',
    'Siedlce', 'Mysłowice', 'Konin', 'Piła', 'Piotrków Trybunalski'
]
polskie_miasta_set = set(polskie_miasta)


def check_risk(tx):
    reasons = []
    score = 0

    def add(points, reason):
        nonlocal score
        score += points
        reasons.append(f"{reason} (+{points})")

    amount = float(tx.get("amount", 0) or 0)
    curr = str(tx.get('currency', 'PLN') or 'PLN').upper()
    balance = float(tx.get('balance', 0) or 0)
    amount_ratio = amount / balance if balance > 0 else 0
    user_id = tx.get("user_id", "unknown")
    location = tx.get("location", "")
    tx_type = tx.get("tx_type", "")

    try:
        tx_time = datetime.fromisoformat(tx["timestamp"])
    except Exception:
        tx_time = datetime.now()
        add(1, "niepoprawny timestamp")

    is_foreign_location = location not in polskie_miasta_set

    # Najsilniejsze sygnały z obecnego generatora fraudów:
    # obca waluta, duża część salda i często zagraniczna lokalizacja.
    if curr != 'PLN':
        add(3, "waluta inna niż PLN")

    if amount_ratio >= 0.75:
        add(2, f"kwota {amount_ratio:.0%} salda")
    elif amount_ratio >= 0.50:
        add(1, f"kwota {amount_ratio:.0%} salda")

    if is_foreign_location:
        add(2, "lokalizacja poza Polską")

    # Te sygnały są słabe: wcześniej generowały dużo fałszywych alarmów,
    # więc same nie powinny oznaczać transakcji jako fraud.
    if tx_type == "transakcja krajowa" and is_foreign_location:
        add(1, "transakcja krajowa z zagranicznej lokalizacji")
    elif tx_type == "transakcja zagraniczna" and not is_foreign_location:
        add(1, "transakcja zagraniczna z polskiej lokalizacji")

    if tx_type in ("wypłata z bankomatu", "zakup online") and amount_ratio >= 0.85:
        add(1, "ryzykowny typ transakcji przy bardzo wysokiej kwocie")

    user_activity[user_id].append(tx_time)
    while user_activity[user_id] and tx_time - user_activity[user_id][0] > timedelta(seconds=60):
        user_activity[user_id].popleft()

    transactions_60s = len(user_activity[user_id])
    if transactions_60s >= 5:
        add(1, f"{transactions_60s} transakcji użytkownika w 60 sekund")

    strong_currency_amount = curr != 'PLN' and amount_ratio >= 0.45
    strong_foreign_amount = is_foreign_location and amount_ratio >= 0.60
    requires_verification = strong_currency_amount or strong_foreign_amount or score >= 5

    if requires_verification:
        return [f"wynik ryzyka {score}/5"] + reasons

    return []


print("Start konsumenta — analiza transakcji bankowych w czasie rzeczywistym...")

try:
    for message in konsument:
        tx = message.value

        risk_reasons = check_risk(tx)

        tx["requires_verification"] = len(risk_reasons) > 0
        tx["risk_reason"] = risk_reasons
        tx["processed_at"] = datetime.now().isoformat()

        if tx["requires_verification"]:
            tx["status"] = "VERIFICATION_REQUIRED"
            tx['fraud_detected'] = 1
        else:
            tx["status"] = "APPROVED"
            tx['fraud_detected'] = 0

        transactions.append(tx)

        ch_row = [
            str(tx.get('tx_id')),
            str(tx.get('user_id')),
            float(tx.get('amount', 0.0)),
            datetime.fromisoformat(tx.get('timestamp')),
            datetime.fromisoformat(tx.get('processed_at')),
            str(tx.get('location', '')),
            str(tx.get('tx_type', '')),
            str(tx.get('currency', 'PLN')),
            float(tx.get('balance', 0.0)),
            int(tx.get('fraud', 0)),
            int(tx.get('fraud_detected', 0)),
            str(tx.get('status', 'APPROVED')),
            tx.get('risk_reason', [])
        ]

        try:
            ch_client.insert(
                table='bank_transactions',
                data=[ch_row],
                database='default',
                column_names=[
                    'tx_id', 'user_id', 'amount', 'timestamp', 'processed_at',
                    'location', 'tx_type', 'currency', 'balance', 'fraud',
                    'fraud_detected', 'status', 'risk_reason'
                ]
            )
        except Exception as ch_err:
            print(f"Błąd zapisu do ClickHouse: {ch_err}")

        if tx["requires_verification"]:
            print(f"ALERT: {tx['tx_id']}")
            print(f"       Użytkownik: {tx['user_id']} ")
            print(f"       Lokalizacja: {tx['location']} | Typ: {tx['tx_type']}")
            print(f"       Kwota: {tx['amount']:>8.2f} {tx['currency']}")
            print(f"       Powód: {tx['risk_reason']} Czy faktyczny fraud: {tx['fraud']==1}")

        time.sleep(1)

except KeyboardInterrupt:
    print("\nZatrzymano zapisywanie.")

finally:
    konsument.close()
    try:
        ch_client.close()
    except NameError:
        pass
    print("Połączenia zamknięte.")

Pomyślnie połączono z ClickHouse.
Start konsumenta — analiza transakcji bankowych w czasie rzeczywistym...
ALERT: BANK-3201501571
       Użytkownik: 6d91ebbd 
       Lokalizacja: Nairobi | Typ: transakcja zagraniczna
       Kwota: 138180.72 USD
       Powód: ['wynik ryzyka 7/5', 'waluta inna niż PLN (+3)', 'kwota 81% salda (+2)', 'lokalizacja poza Polską (+2)'] Czy faktyczny fraud: True
ALERT: BANK-3630831713
       Użytkownik: d9f59e5d 
       Lokalizacja: Kapsztad | Typ: wypłata z bankomatu
       Kwota: 46713.74 EUR
       Powód: ['wynik ryzyka 6/5', 'waluta inna niż PLN (+3)', 'kwota 51% salda (+1)', 'lokalizacja poza Polską (+2)'] Czy faktyczny fraud: True
ALERT: BANK-2657149703
       Użytkownik: a42f59cc 
       Lokalizacja: Dakar | Typ: przelew internetowy
       Kwota: 153585.93 INR
       Powód: ['wynik ryzyka 7/5', 'waluta inna niż PLN (+3)', 'kwota 98% salda (+2)', 'lokalizacja poza Polską (+2)'] Czy faktyczny fraud: True
ALERT: BANK-3509379981
       Użytkownik: c6031e0c 
